# Ragas + OpenAI サンプル

この Notebook は `ragas` の `RubricsScoreWithoutReference` を OpenAI LLM で実行する最小サンプルです。

事前条件:
- `OPENAI_API_KEY` が環境変数または `.env` に設定されている
- 必要なら `RAGAS_OPENAI_MODEL` でモデル名を上書きできる
- 上から順にセルを実行する
- `retrieved_contexts` は任意で、省略しても評価できます

In [ ]:
import os
from pprint import pprint

from dotenv import load_dotenv
from openai import AsyncOpenAI
from ragas.llms import llm_factory

load_dotenv("../.env")

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY を環境変数または .env に設定してください。"
    )

openai_model = os.getenv("RAGAS_OPENAI_MODEL", "gpt-4o-mini")

client_kwargs = {"api_key": openai_api_key}
openai_base_url = os.getenv("OPENAI_BASE_URL")
openai_org = os.getenv("OPENAI_ORG_ID")

if openai_base_url:
    client_kwargs["base_url"] = openai_base_url
if openai_org:
    client_kwargs["organization"] = openai_org

client = AsyncOpenAI(**client_kwargs)
llm = llm_factory(openai_model, client=client)

pprint(
    {
        "model": openai_model,
        "base_url_override": bool(openai_base_url),
        "organization_override": bool(openai_org),
    }
)

In [ ]:
question = "RAG とは何ですか？"
answer = (
    "RAG は Retrieval-Augmented Generation の略です。"
    " 外部知識を検索し、その結果を踏まえて LLM が回答を生成する手法です。"
)
contexts = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It improves answer quality by grounding the model on retrieved documents.",
]

In [ ]:
from ragas.metrics.collections import RubricsScoreWithoutReference

metric = RubricsScoreWithoutReference(llm=llm)

result = await metric.ascore(
    user_input=question,
    response=answer,
    retrieved_contexts=contexts,
)

evaluation = {
    "metric": metric.name,
    "score": result.value,
    "reason": result.reason,
}
pprint(evaluation)